# Create a simple DB here

In [1]:
#create a db with all the resturaunts with their details locally using sqlite and sqlalchemy
# Cell 1: Create a simple DB here
import sqlite3
import json
from pathlib import Path
from sqlalchemy import create_engine, Column, Integer, String, Float, Text, Boolean, JSON
from sqlalchemy.ext.declarative import declarative_base
from sqlalchemy.orm import sessionmaker
from typing import Optional, List

# Create database model
Base = declarative_base()

class Restaurant(Base):
    __tablename__ = 'restaurants'
    
    id = Column(Integer, primary_key=True, autoincrement=True)
    restaurant_id = Column(String(36), unique=True, nullable=False)
    name = Column(String(100), nullable=False)
    cuisine = Column(String(50), nullable=False)
    location = Column(String(100), nullable=False)
    address = Column(Text, nullable=False)
    capacity = Column(Integer, nullable=False)
    price_range = Column(String(10), nullable=False)
    rating = Column(Float, nullable=False)
    features = Column(JSON)
    operating_hours = Column(JSON)
    phone = Column(String(20), nullable=False)
    description = Column(Text)
    is_active = Column(Boolean, default=True)

# Setup database
# Add this to your database setup cell
from datetime import datetime

class Reservation(Base):
    __tablename__ = 'reservations'
    
    id = Column(Integer, primary_key=True, autoincrement=True)
    reservation_id = Column(String(36), unique=True, nullable=False)
    restaurant_id = Column(String(36), nullable=False)  # Links to restaurant
    customer_name = Column(String(100), nullable=False)
    customer_phone = Column(String(20), nullable=False)
    party_size = Column(Integer, nullable=False)
    reservation_date = Column(String(20), nullable=False)  # Store as string for simplicity
    reservation_time = Column(String(10), nullable=False)  # e.g., "19:00"
    special_requests = Column(Text)
    status = Column(String(20), default='confirmed')  # confirmed, cancelled
    created_at = Column(String(30), default=lambda: datetime.now().strftime('%Y-%m-%d %H:%M:%S'))

# Update your database creation


db_path = Path("data/simple_restaurants.db")
db_path.parent.mkdir(exist_ok=True)

engine = create_engine(f"sqlite:///{db_path}", echo=False)
SessionLocal = sessionmaker(bind=engine)
Base.metadata.create_all(bind=engine)

# Load restaurant data
def load_restaurants():
    session = SessionLocal()
    try:
        # Check if data already exists
        if session.query(Restaurant).count() > 0:
            print("✅ Restaurant data already loaded")
            return
        
        # Load from JSON
        with open("data/restaurants.json", 'r', encoding='utf-8') as f:
            restaurants_data = json.load(f)
        
        for restaurant_data in restaurants_data:
            restaurant = Restaurant(**restaurant_data)
            session.add(restaurant)
        
        session.commit()
        print(f"✅ Loaded {len(restaurants_data)} restaurants")
        
    except Exception as e:
        session.rollback()
        print(f"❌ Error loading restaurants: {e}")
    finally:
        session.close()

# Load the data
load_restaurants()


✅ Restaurant data already loaded


C:\Users\sarabjot.s\AppData\Local\Temp\ipykernel_36800\1491387289.py:12: MovedIn20Warning: The ``declarative_base()`` function is now available as sqlalchemy.orm.declarative_base(). (deprecated since: 2.0) (Background on SQLAlchemy 2.0 at: https://sqlalche.me/e/b8d9)
  Base = declarative_base()


# Define tools

In [2]:
from typing import Optional

In [3]:
# Add this after your search_rests_tool function
import uuid
from datetime import datetime, timedelta

def book_reservation_tool(restaurant_name: str, customer_name: str, customer_phone: str, 
                         party_size: int, date: str, time: str, special_requests: str = None):
    """
    Book a reservation at a restaurant
    """
    session = SessionLocal()
    try:
        # Step 1: Find the restaurant
        restaurant = session.query(Restaurant).filter(
            Restaurant.name.ilike(f"%{restaurant_name}%"),
            Restaurant.is_active == True
        ).first()
        
        if not restaurant:
            return f"❌ Restaurant '{restaurant_name}' not found. Please check the name and try again."
        
        # Step 2: Basic validation
        if party_size > restaurant.capacity:
            return f"❌ Sorry, {restaurant.name} has a maximum capacity of {restaurant.capacity} people. Your party size of {party_size} exceeds this limit."
        
        if party_size < 1:
            return f"❌ Party size must be at least 1 person."
        
        # Step 3: Validate phone number (basic)
        if not customer_phone or len(customer_phone.replace('+', '').replace('-', '').replace(' ', '')) < 10:
            return f"❌ Please provide a valid phone number."
        
        # Step 4: Create reservation
        reservation_id = str(uuid.uuid4())[:8].upper()  # Short ID like "A1B2C3D4"
        
        new_reservation = Reservation(
            reservation_id=reservation_id,
            restaurant_id=restaurant.restaurant_id,
            customer_name=customer_name,
            customer_phone=customer_phone,
            party_size=party_size,
            reservation_date=date,
            reservation_time=time,
            special_requests=special_requests,
            status='confirmed'
        )
        
        session.add(new_reservation)
        session.commit()
        
        # Step 5: Generate confirmation
        result = f"🎉 **Reservation Confirmed!**\n\n"
        result += f"📋 **Reservation ID:** {reservation_id}\n"
        result += f"🏪 **Restaurant:** {restaurant.name}\n"
        result += f"📍 **Location:** {restaurant.location}\n"
        result += f"📅 **Date:** {date}\n"
        result += f"🕐 **Time:** {time}\n"
        result += f"👥 **Party Size:** {party_size} people\n"
        result += f"👤 **Name:** {customer_name}\n"
        result += f"📞 **Phone:** {customer_phone}\n"
        
        if special_requests:
            result += f"📝 **Special Requests:** {special_requests}\n"
        
        result += f"\n📍 **Restaurant Details:**\n"
        result += f"   Address: {restaurant.address}\n"
        result += f"   Phone: {restaurant.phone}\n"
        result += f"   Rating: ⭐ {restaurant.rating}/5.0\n"
        
        result += f"\n💡 **Important Notes:**\n"
        result += f"   • Please arrive 10 minutes before your reservation time\n"
        result += f"   • Call {restaurant.phone} if you need to make changes\n"
        result += f"   • Keep your reservation ID: {reservation_id}\n"
        
        print(f"✅ Reservation created: {reservation_id}")
        return result
        
    except Exception as e:
        session.rollback()
        return f"❌ Error creating reservation: {str(e)}. Please try again."
    finally:
        session.close()


In [4]:
# Cell 2: Define tools
def search_rests_tool(cuisine: Optional[str] = None, location: Optional[str] = None, 
                     capacity: Optional[int] = None, features: Optional[str] = None):
    """
    This tool will search for relevant restaurants based on user criteria
    """
    session = SessionLocal()
    try:
        query = session.query(Restaurant).filter(Restaurant.is_active == True)
        
        if cuisine:
            query = query.filter(Restaurant.cuisine.ilike(f"%{cuisine}%"))
        
        if location:
            query = query.filter(Restaurant.location.ilike(f"%{location}%"))
        
        if capacity:
            query = query.filter(Restaurant.capacity >= capacity)
        
        restaurants = query.order_by(Restaurant.rating.desc()).all()
        
        if not restaurants:
            return f"No restaurants found matching your criteria."
        
        result = f"Found {len(restaurants)} restaurants:\n\n"
        for i, restaurant in enumerate(restaurants, 1):
            restaurant_features = ", ".join(restaurant.features[:3]) if restaurant.features else "No special features"
            result += f"{i}. **{restaurant.name}**\n"
            result += f"   📍 Location: {restaurant.location}\n"
            result += f"   🍽️ Cuisine: {restaurant.cuisine}\n"
            result += f"   ⭐ Rating: {restaurant.rating}/5.0\n"
            result += f"   💰 Price: {restaurant.price_range}\n"
            result += f"   👥 Capacity: {restaurant.capacity}\n"
            result += f"   ✨ Features: {restaurant_features}\n"
            result += f"   📞 Phone: {restaurant.phone}\n\n"
        
        print(result)
        return result
        
    except Exception as e:
        return f"Error searching restaurants: {str(e)}"
    finally:
        session.close()



## Define tools schema

In [5]:
from pydantic import BaseModel, Field
# Add this after your GetSearchArgs class
class BookReservationArgs(BaseModel):
    restaurant_name: str = Field(..., description="Name of the restaurant to book")
    customer_name: str = Field(..., description="Customer's full name")
    customer_phone: str = Field(..., description="Customer's phone number")
    party_size: int = Field(..., description="Number of people for the reservation")
    date: str = Field(..., description="Reservation date (e.g., '2025-12-15' or 'tomorrow')")
    time: str = Field(..., description="Reservation time (e.g., '19:00' or '7:00 PM')")
    special_requests: Optional[str] = Field(None, description="Any special requests or dietary requirements")

book_reservation_tool_schema = {
    "type": "function",
    "function": {
        "name": "book_reservation_tool",
        "description": "Book a reservation at a restaurant. Requires restaurant name, customer details, party size, date, and time.",
        "parameters": BookReservationArgs.model_json_schema()
    }
}


In [6]:
# Cell 3: Define tools schema
from pydantic import BaseModel, Field,ValidationError

class GetSearchArgs(BaseModel):
    cuisine: Optional[str] = Field(None, description="The cuisine type the user is looking for (e.g., Italian, Chinese, Indian)")
    location: Optional[str] = Field(None, description="The location where user wants to find restaurants")
    capacity: Optional[int] = Field(None, description="Minimum capacity/party size needed")
    features: Optional[str] = Field(None, description="Special features like 'outdoor seating', 'live music', etc.")

search_rests_tool_schema = {
    "type": "function",
    "function": {
        "name": "search_rests_tool",
        "description": "Gets a list of restaurants pertaining to the user's request based on cuisine, location, capacity, and features.",
        "parameters": GetSearchArgs.model_json_schema()
    }
}


# Setup the LLM

In [ ]:
import os
import getpass

os.environ["HF_TOKEN"] = "hf_...."
from dotenv import load_dotenv
load_dotenv()


In [8]:
from huggingface_hub import InferenceClient

client = InferenceClient(model="moonshotai/Kimi-K2-Thinking")

c:\sarabjot\agent_creation\reservation_system\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [9]:
# Enhanced agent with both search and booking capabilities
system_message = """You are an intelligent restaurant assistant that can both search for restaurants and make reservations.

CAPABILITIES:
1. **Search Restaurants**: Find restaurants by cuisine, location, capacity, features
2. **Book Reservations**: Make reservations at restaurants

WORKFLOW:
1. When users ask for restaurant recommendations, use search_rests_tool
2. When users want to book a restaurant, use book_reservation_tool
3. If search fails, refine your query and try again (up to 3 times)
4. If booking fails, help user understand what went wrong and suggest alternatives

BOOKING REQUIREMENTS:
- Restaurant name (from search results or user specification)
- Customer name and phone number
- Party size (number of people)
- Date and time for reservation
- Optional: special requests

IMPORTANT:
- Always confirm booking details before making reservation
- If information is missing, ask the user to provide it
- Be helpful and friendly throughout the process
- Learn from failures and adapt your approach"""





In [10]:
class IntentResponse(BaseModel):
    """
    Intent classification response for restaurant-related queries.
    
    This model determines whether a user query is related to restaurants and provides
    a confidence score for the classification decision.
    
    Attributes:
        description (str): A clear, concise description of what the query is asking for
        decision (bool): True if the query is restaurant-related, False otherwise
        relevance_score (float): Confidence score between 0.0 and 1.0 indicating how
                                relevant the query is to restaurant services
    
    Examples:
        Restaurant-related query:
        {
            "description": "User is looking for Italian restaurants in a specific location",
            "decision": true,
            "relevance_score": 0.95
        }
        
        Non-restaurant query:
        {
            "description": "User is asking about weather information",
            "decision": false,
            "relevance_score": 0.1
        }
    """
    description: str = Field(
        ..., 
        description="Clear, detailed description of what the user query is requesting. Should be specific and informative.",
    )
    
    decision: bool = Field(
        ..., 
        description="Boolean decision: True if query is related to restaurants (search, booking, recommendations, reviews), False otherwise"
    )
    
    relevance_score: float = Field(
        ..., 
        description="Confidence score between 0.0 and 1.0. Higher scores (0.8-1.0) indicate strong restaurant relevance, lower scores (0.0-0.3) indicate non-restaurant queries",
        ge=0.0,
        le=1.0
    )



In [11]:
intent_system_message = """You are an intelligent intent classification agent specialized in identifying restaurant-related queries.

Your task is to analyze user queries and determine if they are related to restaurant services, then provide a structured response.

CLASSIFICATION CRITERIA:
 Restaurant-related queries include:
- Restaurant search and recommendations
- Making, modifying, or canceling reservations
- Questions about restaurant details (menu, hours, location, reviews)
- Dining preferences and cuisine requests
- Restaurant comparisons and reviews

 Non-restaurant queries include:
- Weather information
- General travel questions (unless specifically about restaurant locations)
- Shopping, entertainment, or other services
- Personal questions unrelated to dining

SCORING GUIDELINES:
- 0.9-1.0: Clearly restaurant-focused (e.g., "Find Italian restaurants", "Book a table")
- 0.7-0.8: Restaurant-related with some ambiguity (e.g., "What's good around here?")
- 0.4-0.6: Potentially restaurant-related but unclear (e.g., "I'm hungry")
- 0.1-0.3: Minimal restaurant connection (e.g., "What time is it?" when context might involve dining)
- 0.0: Completely unrelated to restaurants

RESPONSE FORMAT:
Always respond with exactly three components:
1. Description: A clear, specific explanation of what the user is asking for
2. Decision: True/False based on restaurant relevance
3. Relevance Score: Float between 0.0 and 1.0 based on confidence level

Be precise, consistent, and consider context when available."""


In [12]:
class IntentClassifierAgent:
    def __init__(self, client: InferenceClient, system: str = ""):
        self.client = client
        self.system = system
        self.messages: list = []
        
    def __call__(self, message: str = "") -> IntentResponse:
        """
        Classify user intent and return structured response.
        
        Args:
            message (str): User query to classify
            
        Returns:
            IntentResponse: Structured classification result
        """
        if message:
            self.messages = [{"role": "system", "content": self.system}]  # Reset for each query
            self.messages.append({"role": "user", "content": message})
        
        response_content = self.execute()
        
        try:
            # Parse and validate the response
            response_data = json.loads(response_content)
            return IntentResponse(**response_data)
        except (json.JSONDecodeError, ValidationError) as e:
            # Fallback response for parsing errors
            return IntentResponse(
                description=f"Error parsing response: {str(e)}",
                decision=False,
                relevance_score=0.0
            )
    
    def execute(self) -> str:
        """Execute the intent classification request."""
        completion = self.client.chat.completions.create(
            messages=self.messages,
            max_tokens=300,  # Reduced for more focused responses
            temperature=0.1,  # Lower temperature for more consistent classification
            response_format={
                "type": "json_schema",
                "json_schema": {
                    "name": "IntentResponse",
                    "schema": IntentResponse.model_json_schema(),
                    "strict": True,
                },
            }
        )
        return completion.choices[0].message.content

        

In [13]:
import json

class SelfCorrectingAgent:
    def __init__(self, client: InferenceClient, system: str = "", tools: list = None, max_retries: int = 3):
        self.client = client
        self.system = system
        self.messages: list = []
        self.tools = tools if tools is not None else []
        self.max_retries = max_retries
        if self.system:
            self.messages.append({"role": "system", "content": system})

    def __call__(self, message=""):
        if message:
            self.messages.append({"role": "user", "content": message})

        final_assistant_content = self.execute()

        if final_assistant_content:
            self.messages.append({"role": "assistant", "content": final_assistant_content})

        return final_assistant_content

    def execute(self):
        retry_count = 0
        
        while True:
            completion = self.client.chat.completions.create(
                messages=self.messages,
                tools=self.tools,
                tool_choice="auto"
            )

            response_message = completion.choices[0].message
            print(f"response message is {response_message}\n\n")

            if response_message.tool_calls:
                self.messages.append(response_message)

                tool_outputs = []
                tool_failed = False
                
                for tool_call in response_message.tool_calls:
                    function_name = tool_call.function.name
                    function_args = json.loads(tool_call.function.arguments)

                    # Execute the tool
                    try:
                        if function_name in globals() and callable(globals()[function_name]):
                            function_to_call = globals()[function_name]
                            executed_output = function_to_call(**function_args)
                            tool_output_content = str(executed_output)
                            print(tool_output_content)
                            
                            # Check if tool failed (customize this condition based on your tool's failure patterns)
                            if self.is_tool_failure(tool_output_content):
                                tool_failed = True
                                print(f"🔄 Tool failed (attempt {retry_count + 1}): {function_name}")
                            else:
                                print(f"✅ Tool succeeded: {function_name}")
                                
                        else:
                            tool_output_content = f"Error: Function '{function_name}' not found"
                            tool_failed = True
                            
                    except Exception as e:
                        tool_output_content = f"Error executing {function_name}: {str(e)}"
                        tool_failed = True

                    tool_outputs.append({
                        "tool_call_id": tool_call.id,
                        "role": "tool",
                        "name": function_name,
                        "content": tool_output_content,
                    })

                self.messages.extend(tool_outputs)
                
                # If tool failed and we haven't exceeded retry limit, add refinement instruction
                if tool_failed and retry_count < self.max_retries:
                    retry_count += 1
                    refinement_message = {
                        "role": "user", 
                        "content": f"""The search didn't return results. This might be because:
1. Location name variations (e.g., 'Dilli' instead of 'Delhi', 'CP' instead of 'Connaught Place')
2. Spelling differences or local nicknames
3. Too specific search criteria

Please try again with:
- More common/standard location names
- Broader search terms
- Alternative spellings or variations

Attempt {retry_count}/{self.max_retries}. Try to refine your search parameters."""
                    }
                    self.messages.append(refinement_message)
                    continue  # Go back to LLM for refinement
                
                # If tool succeeded or max retries reached, continue normally
                
            else:
                return response_message.content

    def is_tool_failure(self, tool_output: str) -> bool:
        """
        Determine if the tool execution failed based on output content
        Customize this based on your specific failure patterns
        """
        failure_indicators = [
            "No restaurants found matching your criteria",
            "No restaurants found",
            "Error searching restaurants",
            "Error:",
            "not found"
        ]
        
        return any(indicator.lower() in tool_output.lower() for indicator in failure_indicators)


In [14]:
# Create the self-correcting agent
agent = SelfCorrectingAgent(
    client=client,
    system=system_message,
    tools=[search_rests_tool_schema,book_reservation_tool_schema],
    max_retries=3,
)

# Create the intent classification agent
intent_agent = IntentClassifierAgent(
    client=client,
    system=intent_system_message,
)

In [15]:
# Test the complete workflow
print("🤖 Testing Complete Restaurant Workflow")
print("=" * 60)

# Test queries that demonstrate the full flow
# workflow_tests = [
#     "Find me some good Italian restaurants in Connaught Place",
#     "Book a table at Bella Italia for tomorrow 7 PM for 4 people. My name is Rahul Sharma, phone +91-9876543210",
#     "I want to book Giovanni's for 2 people on 2024-01-20 at 8 PM. Name: Priya Singh, Phone: 9876543211. We need a quiet table please."
# ]
# workflow_tests = ["what were my previous queries?"]
workflow_tests = ["list some nice chinese resturaunts in CP"]


# response = intent_agent(workflow_tests[0])

for query in workflow_tests:
    print(f"\n👤 User: {query}")
    print("-" * 50)
    intent= intent_agent(query)
    if intent.decision:
        response = agent(intent.description)
    print(f"🤖 Agent: {response}")
    print("\n" + "="*60)


🤖 Testing Complete Restaurant Workflow

👤 User: list some nice chinese resturaunts in CP
--------------------------------------------------


HfHubHTTPError: Client error '402 Payment Required' for url 'https://router.huggingface.co/v1/chat/completions' (Request ID: Root=1-6945b810-07b6624e223735ef45f370e1;e0d9582d-9d9a-461a-bfb0-2d15df17a38b)
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/402

You have exceeded your monthly included credits for Inference Providers. Subscribe to PRO to get 20x more monthly included credits.

In [ ]:
response

IntentResponse(description='User is requesting recommendations for good Chinese restaurants located in Connaught Place (CP), New Delhi', decision=True, relevance_score=0.95)

In [ ]:
response.decision

True

In [ ]:
response.description

'User is requesting recommendations for good Chinese restaurants located in Connaught Place (CP), New Delhi'

In [ ]:
from datetime import datetime
datetime.now()

datetime.datetime(2025, 12, 20, 0, 3, 6, 830567)